# Viabilidad de una heladería artesanal italiana en Colombia### Análisis en Python — lectura desde SQL, visualización y exportación para TableauEste notebook forma parte del proyecto de portfolio disponible en:- Repositorio completo: `github.com/GuilleSoler2/Heladeria-Italiana-Colombia`- Dashboard interactivo: Tableau PublicLos datos se leen directamente de la base de datos SQL (`heladeria_colombia.db`) construida en las fases anteriores del proyecto — no se recalculan aquí, se reutilizan, como correspondería en un flujo real de analítica de datos.

## 1. Conexión a la base de datos y lectura del scorecard

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("heladeria_colombia.db")

# Reconstruimos el scorecard con Python, leyendo directamente de la base de datos SQL
df = pd.read_sql_query("SELECT * FROM v_scorecard", conn)
df

## 2. Visualización — Scorecard de ciudadesGráfico de barras horizontales con las 8 ciudades ordenadas por puntuación ponderada, resaltando la ciudad ganadora (Medellín).

In [ ]:
import matplotlib.pyplot as plt

# Ordenamos de mayor a menor score para que el gráfico se lea bien
df_ordenado = df.sort_values("score_total", ascending=True)

plt.figure(figsize=(8, 5))
barras = plt.barh(df_ordenado["ciudad"], df_ordenado["score_total"], color="#1F4E5F")

# Resaltamos la ciudad ganadora (Medellín) en otro color
for barra, ciudad in zip(barras, df_ordenado["ciudad"]):
    if ciudad == "Medellín":
        barra.set_color("#E07A3F")

plt.xlabel("Score total ponderado")
plt.title("Scorecard de ciudades — Heladería artesanal italiana en Colombia")
plt.tight_layout()
plt.show()

## 3. Visualización — Distribución del CAPEX¿En qué se va la inversión inicial? Cargamos la tabla `capex` y la representamos ordenada de mayor a menor.

In [ ]:
# Cargamos el CAPEX y hacemos un gráfico de dónde se va la inversión
df_capex = pd.read_sql_query("SELECT * FROM capex ORDER BY costo_cop DESC", conn)

plt.figure(figsize=(8, 5))
plt.barh(df_capex["concepto"], df_capex["costo_cop"], color="#1F4E5F")
plt.xlabel("Costo (COP)")
plt.title("Distribución del CAPEX — Heladería en Medellín")
plt.gca().invert_yaxis()  # el concepto más caro arriba
plt.tight_layout()
plt.show()

print(f"Inversión total: {df_capex['costo_cop'].sum():,.0f} COP")

Inversión total: 233,351,000 COP

## 4. Exportación de datos para TableauExportamos las tablas clave a CSV. Se generan dos versiones: la estándar (separador `,`) y una segunda (`_v2`, separador `;`) necesaria porque Tableau, en configuración regional en español, espera `;` como separador de columnas en archivos de texto.

In [ ]:
# Exportamos las tablas clave a CSV para usarlas en Tableau
df_scorecard = pd.read_sql_query("SELECT * FROM v_scorecard", conn)
df_scorecard.to_csv("scorecard_ciudades.csv", index=False)

df_capex_export = pd.read_sql_query("SELECT * FROM capex", conn)
df_capex_export.to_csv("capex.csv", index=False)

df_sabores = pd.read_sql_query("SELECT * FROM sabores", conn)
df_sabores.to_csv("sabores.csv", index=False)

print("Archivos CSV creados correctamente")

Archivos CSV creados correctamente

In [ ]:
# Versión v2, con separador ; para que Tableau en español las lea con columnas separadas
df_scorecard.to_csv("scorecard_ciudades_v2.csv", index=False, sep=";")
df_capex_export.to_csv("capex_v2.csv", index=False, sep=";")
df_sabores.to_csv("sabores_v2.csv", index=False, sep=";")

print("Archivos v2 creados con separador ; para Tableau en español")

Archivos v2 creados con separador ; para Tableau en español

## 5. Cierre de la conexión

In [ ]:
conn.close()